In [ ]:
pip install catboost

In [ ]:
import mlflow
import mlflow.catboost
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold

# ==============================================================================
# 0. CONFIGURACIÓN DEL EXPERIMENTO Y VALIDACIÓN CRUZADA
# ==============================================================================
experiment_name = "/modelos-hipervalorizacion-bogota-v2"
mlflow.set_experiment(experiment_name)

N_SPLITS = 5
RANDOM_STATE = 42
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

print("--- 1. CARGA Y PREPARACIÓN DE DATOS ---")
matriz = "/Volumes/workspace/default/datos_bogota/matriz_barrios_unificada.csv"
df = pd.read_csv(matriz)

# Mismo subconjunto supervisado e índice reseteado para garantizar pliegues idénticos
df_supervised = df[df["v_ref_2026"].notnull()].copy().reset_index(drop=True)

y_sup = df_supervised["v_ref_2026"].copy()
cols_a_excluir = [
    "barrio", "geometry", "v_ref_2015", "v_ref_2026",
    "num_listings_airbnb", "area_km2"
]
X_sup = df_supervised.drop(
    columns=[col for col in cols_a_excluir if col in df_supervised.columns]
).copy()

cat_features = ["localidad"]
for col in cat_features:
    if col in X_sup.columns:
        X_sup[col] = X_sup[col].astype(str)

# Mejores hiperparámetros encontrados previamente
params = {
    "iterations": 1500,
    "learning_rate": 0.01,
    "depth": 6,
    "l2_leaf_reg": 5,
    "loss_function": "RMSE",
    "random_seed": RANDOM_STATE,
}


def log_cv_metrics(metricas_dict: dict) -> None:
    """Calcula media, desviación estándar y brecha de sobreajuste, y las registra en MLflow."""
    resumen = {
        "cv_r2_train_mean": float(np.mean(metricas_dict["r2_train"])),
        "cv_r2_train_std": float(np.std(metricas_dict["r2_train"])),
        "cv_r2_val_mean": float(np.mean(metricas_dict["r2_val"])),
        "cv_r2_val_std": float(np.std(metricas_dict["r2_val"])),
        "cv_r2_gap_mean": float(
            np.mean(np.array(metricas_dict["r2_train"]) - np.array(metricas_dict["r2_val"]))
        ),
        "cv_mae_train_mean": float(np.mean(metricas_dict["mae_train"])),
        "cv_mae_val_mean": float(np.mean(metricas_dict["mae_val"])),
        "cv_mae_val_std": float(np.std(metricas_dict["mae_val"])),
        "cv_rmse_val_mean": float(np.mean(metricas_dict["rmse_val"])),
        "cv_rmse_val_std": float(np.std(metricas_dict["rmse_val"])),
    }
    mlflow.log_metrics(resumen)
    for k, v in resumen.items():
        print(f"  {k}: {v:,.4f}")


def ejecutar_experimento_catboost(run_name: str, artifact_path: str, usar_log: bool = False) -> None:
    """Ejecuta validación cruzada K-Fold, registra métricas y empaqueta el modelo final."""
    print(f"\n--- EVALUANDO {run_name} ---")
    metricas = {
        "r2_train": [], "r2_val": [],
        "mae_train": [], "mae_val": [], "rmse_val": []
    }

    for train_idx, val_idx in kf.split(X_sup):
        X_tr, X_val = X_sup.iloc[train_idx], X_sup.iloc[val_idx]
        y_tr, y_val = y_sup.iloc[train_idx], y_sup.iloc[val_idx]

        y_tr_fit = np.log(y_tr) if usar_log else y_tr

        modelo_fold = CatBoostRegressor(**params, verbose=0)
        modelo_fold.fit(X_tr, y_tr_fit, cat_features=cat_features)

        y_pred_tr = modelo_fold.predict(X_tr)
        y_pred_val = modelo_fold.predict(X_val)

        if usar_log:
            y_pred_tr = np.exp(y_pred_tr)
            y_pred_val = np.exp(y_pred_val)

        metricas["r2_train"].append(r2_score(y_tr, y_pred_tr))
        metricas["r2_val"].append(r2_score(y_val, y_pred_val))
        metricas["mae_train"].append(mean_absolute_error(y_tr, y_pred_tr))
        metricas["mae_val"].append(mean_absolute_error(y_val, y_pred_val))
        metricas["rmse_val"].append(np.sqrt(mean_squared_error(y_val, y_pred_val)))

    with mlflow.start_run(run_name=run_name):
        mlflow.log_params({
            **params,
            "modelo": "CatBoostRegressor",
            "features": "todas_con_v_ref_2025",
            "target_transform": "Logaritmo Natural" if usar_log else "Ninguna",
            "cv_splits": N_SPLITS,
            "random_state": RANDOM_STATE,
        })

        log_cv_metrics(metricas)

        # Entrenamiento final sobre todo el conjunto supervisado para empaquetar el artefacto
        y_final_fit = np.log(y_sup) if usar_log else y_sup
        modelo_final = CatBoostRegressor(**params, verbose=0)
        modelo_final.fit(X_sup, y_final_fit, cat_features=cat_features)

        mlflow.catboost.log_model(
            cb_model=modelo_final,
            artifact_path=artifact_path,
            input_example=X_sup.head(5),
        )


# ==============================================================================
# RUN 1: CATBOOST SIN LOGARITMO (Precios Brutos)
# ==============================================================================
ejecutar_experimento_catboost(
    run_name="CatBoost_Sin_Log",
    artifact_path="catboost_model_bruto",
    usar_log=False,
)

# ==============================================================================
# RUN 2: CATBOOST CON LOGARITMO
# ==============================================================================
ejecutar_experimento_catboost(
    run_name="CatBoost_Con_Log",
    artifact_path="catboost_model_log",
    usar_log=True,
)